# ETL Pipeline - Raw Data to Normalized Tables
Extract raw Kaggle CSVs → clean ingredients → normalize to 3NF → export processed CSVs

In [1]:
import pandas as pd

In [2]:
products = pd.read_csv("../data/raw/product_info.csv")
reviews = pd.read_csv("../data/raw/sephora_full.csv", low_memory=False)

In [3]:
print(products.shape)
print(reviews.shape)

(8494, 27)
(1094411, 18)


In [4]:
print(products.columns)
print(reviews.columns)

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')
Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')


In [5]:
products_ing = products[["product_id", "ingredients"]].copy()
reviews_skin = reviews[["product_id", "skin_type"]].copy()

In [6]:
df_map = reviews_skin.merge(
    products_ing,
    on="product_id",
    how="left"
)

df_map.head()

,product_id,skin_type,ingredients
0,P504322,dry,"['Water (Aqua), Dipropylene Glycol, Peg-6 Capr..."
1,P420652,NaN,"['Diisostearyl Malate, Hydrogenated Polyisobut..."
2,P420652,dry,"['Diisostearyl Malate, Hydrogenated Polyisobut..."
3,P420652,combination,"['Diisostearyl Malate, Hydrogenated Polyisobut..."
4,P420652,combination,"['Diisostearyl Malate, Hydrogenated Polyisobut..."


Above this caused the partial breaks in the ingredient cleaning and was not split clean

In [7]:
df_map["ingredients_clean"] = (
    df_map["ingredients"]
    .fillna("")
    .str.lower()
    .str.replace(r"[\[\]\n\r]", "", regex=True)   # remove brackets
    .str.replace("'", "", regex=False)            #remove quotes
    .str.replace(";", ",")                        #unify separators
)


In [8]:
df_map["ingredients_list"] = df_map["ingredients_clean"].str.split(",")

In [9]:
df_exploded = df_map.explode("ingredients_list")

df_exploded["ingredient"] = (
    df_exploded["ingredients_list"]
    .str.strip()
)

# Remove empty or numeric junk
df_exploded = df_exploded[
    (df_exploded["ingredient"] != "") &
    (~df_exploded["ingredient"].str.isnumeric())
]

df_exploded = df_exploded.drop(columns=["ingredients", "ingredients_clean", "ingredients_list"])

In [10]:
df_exploded.to_csv(
    "../data/processed/product_ingredient_skin.csv",
    index=False
)
print("Saved processed mapping.")


Saved processed mapping.


In [11]:
df_exploded.head()

,product_id,skin_type,ingredient
0,P504322,dry,water (aqua)
0,P504322,dry,dipropylene glycol
0,P504322,dry,peg-6 caprylic/capric glycerides
0,P504322,dry,glycerin
0,P504322,dry,2-hexanediol


In [12]:
df_exploded.shape

(35789146, 3)

Duplicates !!!!!!

In [13]:
df_exploded["ingredient"].nunique()

8156

In [14]:
df_exploded["skin_type"].unique()

array(['dry', nan, 'combination', 'normal', 'oily'], dtype=object)

In [15]:
df_product_ing = (
    df_exploded[["product_id", "ingredient"]]
    .dropna()
    .drop_duplicates()
)

df_product_ing.shape

(81486, 2)

* ~8.4K products
* ~81K product–ingredient relationships
* Average ≈ 9–10 ingredients per product

In [16]:
df_product_skin = (
    df_exploded[["product_id", "skin_type"]]
    .dropna()
    .drop_duplicates()
)

df_product_skin.shape

(8408, 2)

* ~8.4K unique products
* each product mapped to one or more skin types

In [17]:
df_product_ing.to_csv(
    "../data/processed/product_ingredients.csv",
    index=False
)

df_product_skin.to_csv(
    "../data/processed/product_skin_types.csv",
    index=False
)

print("Normalized bridge tables saved.")


Normalized bridge tables saved.


In [18]:
#CHECK
df_product_ing["ingredient"].isna().sum()
df_product_skin["skin_type"].isna().sum()

np.int64(0)

In [19]:
ingredients = (
    df_product_ing["ingredient"]
    .drop_duplicates()
    .reset_index(drop=True)
    .to_frame(name="ingredient_name")
)

ingredients["ingredient_id"] = ingredients.index + 1

ingredients.head()


,ingredient_name,ingredient_id
0,water (aqua),1
1,dipropylene glycol,2
2,peg-6 caprylic/capric glycerides,3
3,glycerin,4
4,2-hexanediol,5


In [20]:
product_ingredients = df_product_ing.merge(
    ingredients,
    left_on="ingredient",
    right_on="ingredient_name",
    how="left"
)[["product_id", "ingredient_id"]]

product_ingredients.head()

,product_id,ingredient_id
0,P504322,1
1,P504322,2
2,P504322,3
3,P504322,4
4,P504322,5


In [21]:
products_sql = products[[
    "product_id",
    "product_name",
    "brand_name",
    "price_usd",
    "primary_category"
]].drop_duplicates()

In [22]:
reviews_sql = reviews[[
    "author_id",
    "product_id",
    "rating",
    "skin_type",
    "review_text"
]].copy()

reviews_sql["review_id"] = range(1, len(reviews_sql) + 1)

In [23]:
ingredients.to_csv("../data/processed/ingredients.csv", index=False)
product_ingredients.to_csv("../data/processed/product_ingredients.csv", index=False)  # overwrite old one
products_sql.to_csv("../data/processed/products.csv", index=False)
reviews_sql.to_csv("../data/processed/reviews.csv", index=False)

print("SQL tables created successfully.")


SQL tables created successfully.


* ingredients.csv - dimension table
* products.csv - dimension table
* reviews.csv - fact table
* product_ingredients.csv - bridge table
* product_skin_types.csv - auxiliary bridge
* product_ingredient_skin.csv - intermediate